# Moving between R and Python

This image has one Python interpreter (`/opt/venv`), and `RETICULATE_PYTHON` points at it.
So the Python kernel in JupyterLab, `reticulate` from R, and anything you `pip install`
later are all the same environment — you are not maintaining two of them.

This notebook runs on the **R** kernel and calls Python from inside it.

In [ ]:
library(reticulate)
py_config()

## Same model, both languages

Fit the same regression with R's `lm` and with `statsmodels`, then compare coefficients.
Agreement to ~1e-13 is the arithmetic agreeing; it is a useful check that data crossed the
boundary intact rather than being silently reordered or coerced.

In [ ]:
sm <- import("statsmodels.api")

r_fit <- lm(mpg ~ wt + hp, data = mtcars)

# Build the design matrix in R and hand over a numeric array. statsmodels'
# formula API (smf$ols) inspects the calling Python stack frame, which does not
# exist when the caller is R, so it fails under reticulate.
X <- model.matrix(~ wt + hp, data = mtcars)
py_fit <- sm$OLS(mtcars$mpg, X)$fit()

data.frame(
  term = c("(Intercept)", "wt", "hp"),
  R = unname(coef(r_fit)),
  statsmodels = as.numeric(py_fit$params)
)

In [ ]:
max_diff <- max(abs(unname(coef(r_fit)) - as.numeric(py_fit$params)))
cat(sprintf("largest coefficient difference: %.2e\n", max_diff))
stopifnot(max_diff < 1e-8)

## Handing over a data frame

`r_to_py()` converts an R data frame to a pandas DataFrame in memory. Convenient for small
data, expensive for large data — it copies.

Note what comes back: `py_to_r()` turns the pandas result into an R data frame, and pandas'
row labels become R row names, so you subset it with ordinary R syntax.

In [ ]:
pd_df <- r_to_py(mtcars)
cat(class(pd_df)[1], "\n")

described <- py_to_r(pd_df$describe())
described[c("mean", "std"), c("mpg", "wt", "hp")]

## Conversion is on by default

`import("pandas")` marks the module for automatic conversion: every value it returns is
converted to the nearest R type on the way out. That is why the next cell asks for
`convert = FALSE` — without it `read_parquet()` would hand back an R data frame, and the
pandas methods below would not exist on it.

Reach for `convert = FALSE` whenever you want to chain Python methods, and call `py_to_r()`
explicitly at the point you want R types back.

## Handing over a large data frame

For anything big, write Parquet with `arrow` and read it from Python. Both sides speak the
same on-disk format, the file lands in `/work` where it persists, and neither side has to
hold a second copy in memory.

In [ ]:
library(arrow)

big <- data.frame(
  g = rep(letters[1:5], each = 20000),
  x = rnorm(100000),
  y = rnorm(100000)
)
path <- "/work/interop-demo.parquet"
write_parquet(big, path)
cat(sprintf("wrote %s (%.1f MB)\n", path, file.size(path) / 1e6))

In [ ]:
pd <- import("pandas", convert = FALSE)
back <- pd$read_parquet(path)
cat(class(back)[1], "\n")

py_means <- py_to_r(back$groupby("g")$x$mean())
r_means  <- tapply(big$x, big$g, mean)

print(rbind(python = py_means, R = r_means[names(py_means)]))
stopifnot(max(abs(py_means - r_means[names(py_means)])) < 1e-12)
cat("group means agree\n")

## scikit-learn from R

Nothing stops you using a Python model as one more R function. Here a gradient-boosting
regressor is trained from R and scored against R's own linear model on held-out rows.

In [ ]:
skl <- import("sklearn.ensemble")
sklm <- import("sklearn.metrics")

set.seed(42)
train <- sample(nrow(mtcars), 24)
features <- c("wt", "hp", "disp")

gbm <- skl$GradientBoostingRegressor(random_state = 0L)
gbm$fit(as.matrix(mtcars[train, features]), mtcars$mpg[train])
pred_py <- gbm$predict(as.matrix(mtcars[-train, features]))

lm_fit <- lm(mpg ~ wt + hp + disp, data = mtcars[train, ])
pred_r <- predict(lm_fit, newdata = mtcars[-train, ])

rmse <- function(a, b) sqrt(mean((a - b)^2))
truth <- mtcars$mpg[-train]
cat(sprintf("RMSE  sklearn GBM: %.2f\n      R lm         : %.2f\n",
            rmse(pred_py, truth), rmse(pred_r, truth)))

On 24 training rows neither model is meaningfully better — the point is that both live in
one workspace, so the comparison costs nothing to run.

## Going the other direction

From the Python kernel, `rpy2` is not installed; the usual route is Parquet again, or run
the R work on the R kernel and exchange files through `/work`. Reticulate is a one-way
door by design: R drives Python, not the reverse.

In [ ]:
unlink(path)